# Section 1: Environment Setup & GPU Check

Installs core dependencies (`ultralytics`, `roboflow`, `opencv-python`, etc.) and verifies PyTorch CUDA hardware acceleration.

In [ ]:
!pip install ultralytics roboflow opencv-python matplotlib pyyaml --quiet

In [ ]:
import os
import gc
import glob
import random
import shutil
import cv2
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from IPython.display import Image as IPImage, display
from roboflow import Roboflow
from ultralytics import YOLO, RTDETR

# Environment & Hardware Verification
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available? {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  WARNING: Running on CPU. Ensure GPU acceleration is enabled for training.")

# Section 2: Dataset Acquisition & Verification

Downloads version 4 of the public Roboflow lettuce disease dataset or loads an existing local directory, followed by image and label integrity checks across dataset splits.

In [ ]:
# --- OPTION A: Download via Roboflow ---
API_KEY_ROBOFLOW = "<API-KEY-ROBOFLOW>" # Replace with your Roboflow API key
dataset_folder = "disease-lettuce-4"

if not os.path.exists(dataset_folder):
    rf = Roboflow(api_key=API_KEY_ROBOFLOW)
    project = rf.workspace("yssp").project("disease-lettuce-h69zb")
    version = project.version(4)
    dataset = version.download("yolov8")
    yaml_path = os.path.join(dataset.location, "data.yaml")
    print(f"✅ Dataset successfully downloaded to: {dataset.location}")
else:
    # --- OPTION B: Load existing local dataset ---
    print(f"✅ Using existing local dataset at: {dataset_folder}")
    class LocalDataset:
        location = os.path.abspath(dataset_folder)
    dataset = LocalDataset()
    yaml_path = os.path.join(dataset.location, "data.yaml")

# --- DATASET DISTRIBUTION CHECK ---
splits = ['train', 'valid', 'test']
counts = {}

print("\n" + "=" * 60)
print("📊 ORIGINAL DATASET DISTRIBUTION CHECK")
print("=" * 60)

for split in splits:
    images_path = os.path.join(dataset.location, split, 'images')
    labels_path = os.path.join(dataset.location, split, 'labels')

    if os.path.exists(images_path):
        images = [f for f in os.listdir(images_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        labels = [f for f in os.listdir(labels_path) if f.endswith('.txt')] if os.path.exists(labels_path) else []
        
        counts[split] = {'images': len(images), 'labels': len(labels)}

        print(f"\n📁 Split {split.upper()}:")
        print(f"   📸 Images: {len(images)} | 🏷️  Labels: {len(labels)}")
        if len(images) != len(labels):
            print(f"   ⚠️  WARNING: Imbalance detected! {abs(len(images) - len(labels))} file(s) without matching labels.")
    else:
        print(f"\n📁 Split {split.upper()}: ❌ Directory not found")
        counts[split] = {'images': 0, 'labels': 0}

total_images = sum([counts[s]['images'] for s in splits])
total_labels = sum([counts[s]['labels'] for s in splits])

print("\n" + "=" * 60)
print(f"📦 OVERALL ORIGINAL TOTAL: {total_images} images | {total_labels} labels")
print("=" * 60)

# Section 3: Dataset Rebalancing (80% Train / 20% Test)

Pools all annotated images, applies a random seed-based shuffle (`seed=42`), splits the data 80% for training and 20% for validation/testing, and rebuilds the directory tree and `data.yaml`

In [ ]:
# --- OPTION: Load existing rebalanced dataset if present ---
dataset_rebalanced_folder = "disease-lettuce-4_rebalanced"

if os.path.exists(dataset_rebalanced_folder):
    print(f"✅ Rebalanced dataset found at: {dataset_rebalanced_folder}")
    class RebalancedDataset:
        location = os.path.abspath(dataset_rebalanced_folder)
    dataset = RebalancedDataset()
    yaml_path = os.path.join(dataset.location, "data.yaml")
else:
    # --- REBALANCING (80/20) ---
    random.seed(42)

    print("=" * 60)
    print("⚖️  REBALANCING DATASET (80% Train / 20% Test)")
    print("=" * 60)

    all_data = []

    for split in ['train', 'valid', 'test']:
        images_path = os.path.join(dataset.location, split, 'images')
        labels_path = os.path.join(dataset.location, split, 'labels')

        if os.path.exists(images_path):
            for img_file in os.listdir(images_path):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    img_path = os.path.join(images_path, img_file)
                    label_file = os.path.splitext(img_file)[0] + '.txt'
                    label_path = os.path.join(labels_path, label_file)

                    if os.path.exists(label_path):
                        all_data.append({'image': img_path, 'label': label_path, 'filename': img_file})

    random.shuffle(all_data)

    split_idx = int(len(all_data) * 0.8)
    train_data = all_data[:split_idx]
    test_data = all_data[split_idx:]

    new_dataset_path = os.path.abspath(dataset_rebalanced_folder)

    if os.path.exists(new_dataset_path):
        shutil.rmtree(new_dataset_path)

    for split in ['train', 'test']:
        os.makedirs(os.path.join(new_dataset_path, split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(new_dataset_path, split, 'labels'), exist_ok=True)

    for item in train_data:
        shutil.copy2(item['image'], os.path.join(new_dataset_path, 'train', 'images', item['filename']))
        label_file = os.path.splitext(item['filename'])[0] + '.txt'
        shutil.copy2(item['label'], os.path.join(new_dataset_path, 'train', 'labels', label_file))

    for item in test_data:
        shutil.copy2(item['image'], os.path.join(new_dataset_path, 'test', 'images', item['filename']))
        label_file = os.path.splitext(item['filename'])[0] + '.txt'
        shutil.copy2(item['label'], os.path.join(new_dataset_path, 'test', 'labels', label_file))

    original_yaml = os.path.join(dataset.location, 'data.yaml')
    new_yaml = os.path.join(new_dataset_path, 'data.yaml')

    if os.path.exists(original_yaml):
        with open(original_yaml, 'r') as f:
            data_config = yaml.safe_load(f)

        data_config['path'] = new_dataset_path
        data_config['train'] = 'train/images'
        data_config['val'] = 'test/images'
        data_config['test'] = 'test/images'

        with open(new_yaml, 'w') as f:
            yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

    class RebalancedDataset:
        location = new_dataset_path
    dataset = RebalancedDataset()
    yaml_path = new_yaml

    print("✅ Dataset rebalancing completed successfully!")
    print(f"📂 New dataset directory: {new_dataset_path}")
    print(f"📄 New YAML config file: {yaml_path}")

# Section 4: Pre-trained Weights Download

Downloads pre-trained COCO base weights for YOLO11-L and RT-DETR-L.

In [ ]:
models_to_download = [
    ("YOLO11l", "yolo11l.pt"),
    ("RT-DETR-l", "rtdetr-l.pt")
]

for name, file_name in models_to_download:
    if os.path.exists(file_name):
        print(f"✅ {name}: Weights file '{file_name}' already exists locally.")
    else:
        print(f"⏳ {name}: Downloading '{file_name}'...")
        try:
            if "rtdetr" in file_name:
                RTDETR(file_name)
            else:
                YOLO(file_name)
            print(f"   -> {name} download completed successfully!")
        except Exception as e:
            print(f"❌ Error downloading {name}: {e}")

# Section 5: Model Training (512 px, Batch 8)

Fine-tunes YOLO11-L and RT-DETR-L models at 512 px resolution with batch size 8, cache enabled, and automatic mixed precision (AMP).

In [ ]:
torch.backends.cudnn.benchmark = False

TRAIN_ARGS = {
    'data': yaml_path,     
    'epochs': 50,
    'patience': 25,
    'imgsz': 512,            
    'batch': 8,              # Batch size per training run
    'device': 0,
    'workers': 0,          
    'project': 'treinamento_lettuce_results',
    'seed': 42,
    'save': True,
    'plots': True,
    'exist_ok': True,
    'amp': True,
    'cache': True            # Accelerated image loading
}

models = [
    {'name': 'YOLO11',    'file': 'yolo11l.pt', 'cls': YOLO},
    {'name': 'RT-DETR-l', 'file': 'rtdetr-l.pt', 'cls': RTDETR}
]

print("🚀 Starting model training routine at 512 px (Batch 8)...")

for i, config in enumerate(models):
    print(f"\n{'='*60}")
    print(f"({i+1}/{len(models)}) TRAINING MODEL: {config['name']}")
    print(f"{'='*60}")

    try:
        model = config['cls'](config['file'])
        results = model.train(**TRAIN_ARGS, name=config['name'])
        print(f"✅ Training completed successfully for {config['name']}!")
    except Exception as e:
        print(f"❌ Critical error during training of {config['name']}: {e}")

    # Memory Cleanup
    if 'model' in locals(): del model
    if 'results' in locals(): del results
    gc.collect()
    torch.cuda.empty_cache()
    print("🧹 GPU memory cleared.\n")

# Section 6: Validation & Performance Metrics (Protocols A & B)

Evaluates best checkpoints (`best.pt`) under two protocols:   
- Protocol A (Paper Benchmark): 512 px resolution, `conf=0.001`, `iou=0.7`.
- Protocol B (Filtered Protocol): 640 px resolution, `conf=0.25`, `iou=0.6`. 

In [ ]:
models_eval = [
    {'name': 'YOLO11',    'path': os.path.join('treinamento_lettuce_results', 'YOLO11', 'weights', 'best.pt'),    'cls': YOLO},
    {'name': 'RT-DETR-l', 'path': os.path.join('treinamento_lettuce_results', 'RT-DETR-l', 'weights', 'best.pt'), 'cls': RTDETR}
]

for item in models_eval:
    print(f"\n{'='*60}")
    print(f"📊 METRICS EVALUATION: {item['name']}")
    print(f"{'='*60}")
    
    if os.path.exists(item['path']):
        model = item['cls'](item['path'])
        
        # --- PROTOCOL A (512 px, conf=0.001, iou=0.7) ---
        print("\n🔹 Protocol A (Standard Benchmark Protocol - 512 px):")
        metrics_a = model.val(
            data=yaml_path,
            split='test',
            imgsz=512,
            batch=4,
            conf=0.001,
            iou=0.7,
            plots=True,
            save_json=False,
            workers=0
        )
        print(f"   • mAP50:     {metrics_a.box.map50:.4f}")
        print(f"   • mAP50-95:  {metrics_a.box.map:.4f}")
        print(f"   • Precision: {metrics_a.box.mp:.4f}")
        print(f"   • Recall:    {metrics_a.box.mr:.4f}")

        # --- PROTOCOL B (640 px, conf=0.25, iou=0.6) ---
        print("\n🔹 Protocol B (Filtered Protocol - 640 px):")
        metrics_b = model.val(
            data=yaml_path,
            split='test',
            imgsz=640,
            batch=4,
            conf=0.25,
            iou=0.6,
            plots=False,
            save_json=False,
            workers=0
        )
        print(f"   • mAP50:     {metrics_b.box.map50:.4f}")
        print(f"   • mAP50-95:  {metrics_b.box.map:.4f}")
        print(f"   • Precision: {metrics_b.box.mp:.4f}")
        print(f"   • Recall:    {metrics_b.box.mr:.4f}")
    else:
        print(f"⚠️ Weights file not found at: {item['path']}")

# Section 7: Image & Multi-Resolution Video Inference

Runs inference on test images and on micro-UAV field video clips across both 512 px and 1280 px resolutions.

In [ ]:
import os
import gc
import torch
from pathlib import Path
from ultralytics import YOLO, RTDETR

# --- 1. TEST IMAGE INFERENCE ---
test_images_path = os.path.join(dataset.location, "test", "images")

models_to_predict = [
    {'name': 'YOLO11',    'path': os.path.join('treinamento_lettuce_results', 'YOLO11',    'weights', 'best.pt'), 'cls': YOLO,   'iou': 0.45},
    {'name': 'RT-DETR-l', 'path': os.path.join('treinamento_lettuce_results', 'RT-DETR-l', 'weights', 'best.pt'), 'cls': RTDETR, 'iou': 0.65}
]

print("📸 Running predictions on test images...")
for item in models_to_predict:
    if os.path.exists(item['path']):
        print(f"\n--- Images: {item['name']} ---")
        model = item['cls'](item['path'])
        results_img = model.predict(
            source=test_images_path,
            conf=0.35,
            iou=0.5,
            save=True,
            imgsz=640,
            device=0,
            project='runs/detect',
            name='predict',
            exist_ok=True
        )
        print(f"✅ Predictions saved to: {results_img[0].save_dir}")
        del model
        gc.collect()
        torch.cuda.empty_cache()
    else:
        print(f"⚠️ Weights not found at: {item['path']}")

# --- 2. MULTI-RESOLUTION VIDEO INFERENCE (512 px & 1280 px) ---
input_folder = Path('videos')
video_files = list(input_folder.glob('*.mp4')) + list(input_folder.glob('*.MP4'))

if video_files:
    print(f"\n🎬 Found {len(video_files)} video file(s) for processing.")
    
    for item in models_to_predict:
        if os.path.exists(item['path']):
            model = item['cls'](item['path'])
            
            for res_eval in [512, 1280]:
                print(f"\n" + "="*60)
                print(f"🎥 PROCESSING VIDEOS | Model: {item['name']} | Resolution: {res_eval} px")
                print("="*60)
                
                for video_path in video_files:
                    print(f" Processing: {video_path.name}")
                    results = model.predict(
                        source=str(video_path),
                        save=True,
                        imgsz=res_eval,
                        conf=0.25,
                        iou=item['iou'],
                        device=0,
                        stream=True
                    )
                    for _ in results: pass
            
            del model
            gc.collect()
            torch.cuda.empty_cache()
    print("\n✅ Video processing completed for BOTH models!")
else:
    print("\nℹ️ No video files found inside 'videos/' directory.")

# Section 8: Visual Inspection & Ground Truth Comparison

Displays generated prediction outputs from `runs/detect/predict` and compares ground-truth boxes side-by-side with model detections.

In [ ]:
results_folder = 'runs/detect/predict'
predicted_images = glob.glob(f'{results_folder}/*.jpg')

# 1. Display top predicted samples
if predicted_images:
    print(f"Displaying top 5 prediction samples from: {results_folder}\n")
    for img_path in predicted_images[:5]:
        display(IPImage(filename=img_path, width=600))
else:
    print("No predicted images found in the target directory.")

# 2. Side-by-Side Visual Comparison (Ground Truth vs Model Prediction)
if predicted_images and 'yaml_path' in locals() and os.path.exists(yaml_path):
    with open(yaml_path, 'r') as file:
        data_info = yaml.safe_load(file)
        class_names = data_info.get('names', [])

    image_index = 0
    predicted_image_path = predicted_images[image_index]
    predicted_image = Image.open(predicted_image_path)
    base_filename = os.path.basename(predicted_image_path)

    original_image_path = os.path.join(dataset.location, "test", "images", base_filename)
    label_path = os.path.join(dataset.location, "test", "labels", os.path.splitext(base_filename)[0] + ".txt")

    if os.path.exists(original_image_path):
        original_image = cv2.imread(original_image_path)
        h, w, _ = original_image.shape
        ground_truth_image = original_image.copy()

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    class_id, x_center, y_center, width, height = map(float, line.split())
                    class_id = int(class_id)
                    x1 = int((x_center - width / 2) * w)
                    y1 = int((y_center - height / 2) * h)
                    x2 = int((x_center + width / 2) * w)
                    y2 = int((y_center + height / 2) * h)

                    cv2.rectangle(ground_truth_image, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    label_name = class_names[class_id] if class_id < len(class_names) else str(class_id)
                    cv2.putText(ground_truth_image, label_name, (x1, max(y1 - 10, 20)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        ground_truth_image = cv2.cvtColor(ground_truth_image, cv2.COLOR_BGR2RGB)

        fig, ax = plt.subplots(1, 2, figsize=(16, 8))
        ax[0].imshow(ground_truth_image)
        ax[0].set_title(f'Ground Truth\n{base_filename}', fontsize=12)
        ax[0].axis('off')

        ax[1].imshow(predicted_image)
        ax[1].set_title('Model Prediction', fontsize=12)
        ax[1].axis('off')

        plt.tight_layout()
        plt.show()